# Fluxo de tranformações das bases

Vamos organizar todo o fluxo de transformações aplicadas as bases, abstraindo os códigos para funções.

## 1 Uma função para cada base

In [0]:
def get_files_content(path):
    df_l = spark.read.csv(f"{path}/logs_streaming.csv", header=True, inferSchema=True)
    df_f = spark.read.csv(f"{path}/catalogo_filmes.csv", header=True, inferSchema=True)
    df_u = spark.read.csv(f"{path}/usuarios.csv", header=True, inferSchema=True)

    print(f"Logs: ({df_l.count()}, {len(df_l.columns)})\nFilmes: ({df_f.count()}, {len(df_f.columns)})\nUsuários: ({df_u.count()}, {len(df_u.columns)})")

    return df_l, df_f, df_u

files_path                      = "/Volumes/academy/dados-streaing-filmes"
df_logs, df_filmes, df_usuarios = get_files_content(files_path)

Logs: (1007, 10)
Filmes: (62, 7)
Usuários: (82, 6)


### 1.1 Tratamentos na base de Logs

In [0]:
import pyspark.sql.functions as f
from itertools import chain

def to_date_multi_format(
    coluna, 
    formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS", "yyyy/dd/MM")
):
    return f.coalesce(*[f.try_to_date(coluna, fmt) for fmt in formatos_suportados])


def transform_logs(df):
    mapeamento_status  = {"complete": "completed", "erro": "error"}
    mapa_expr_status   = f.create_map([f.lit(x) for x in chain(*mapeamento_status.items())])
    mapeamento_country = {"brazil": "brasil", "br": "brasil", "colômbia": "colombia", "spain": "espanha", "méxico": "mexico", "united states": "usa"}
    mapa_expr_country  = f.create_map([f.lit(x) for x in chain(*mapeamento_country.items())])

    return (
        df
        .dropDuplicates(subset=['session_id', 'user_id', 'movie_id'])
        .withColumn('watch_time_minutes', f.expr('try_cast(watch_time_minutes as float)').cast('int'))
        .withColumn('watch_time_minutes', f.when(f.col("watch_time_minutes") > 0, f.col("watch_time_minutes")).otherwise(0))
        .withColumn("watch_time_hours", f.round(f.col("watch_time_minutes")/60, 2))
        .withColumn("watch_category", f.when((f.col("watch_time_minutes") >= 120) & (df.watch_time_minutes.isNotNull()), "Longo")
                    .otherwise(f.when(f.col("watch_time_minutes") < 120, "Curto").otherwise(None)))
        .withColumn("playback_status", f.lower(f.trim(f.col("playback_status"))))
        .withColumn("playback_status", f.when(f.col("playback_status").isin(list(mapeamento_status.keys())), mapa_expr_status[f.col("playback_status")])
                    .otherwise(f.col("playback_status")))
        .withColumn("subscription_type", f.lower("subscription_type"))
        .withColumn("country", f.lower(f.trim(f.col("country"))))
        .withColumn("country", f.when(f.col("country").isin(list(mapeamento_country.keys())), mapa_expr_country[f.col("country")]).otherwise(f.col("country")))
        .withColumn("device", f.lower(f.trim(f.col("device"))))
        .withColumn('device', f.regexp_replace('device', '_', ' '))
        .withColumn('rating', f.expr('try_cast(rating as float)').cast('int'))
        .withColumn('rating', f.when(f.col("rating") > 5, 5).otherwise(f.col("rating")))
        .withColumn("watch_date", to_date_multi_format(f.split(df["watch_date"], ' ')[0]))
        .dropna(subset=['user_id']).dropna(subset=['movie_id']).dropna(subset=['session_id'])
        .fillna({"country": "desconhecido", "subscription_type": "unknown"})
        .fillna("sem informacao")
    )

In [0]:
# df_logs = spark.read.csv(f"{files_path}/logs_streaming.csv", header=True, inferSchema=True)
df_logs.filter(df_logs.session_id.isin('SFCF6C0D030', 'S228044CE8C', 'S2A082D3FBC', 'S74DAA8499B')).sort(df_logs.session_id).show()
df_logs.printSchema()

+-----------+-------+--------+--------+---------+-----------------+------------------+----------+------+---------------+
| session_id|user_id|movie_id|  device|  country|subscription_type|watch_time_minutes|watch_date|rating|playback_status|
+-----------+-------+--------+--------+---------+-----------------+------------------+----------+------+---------------+
|S228044CE8C|  U0036|   M0024|Smart TV|  Espanha|             free|                26|25/02/2026|     2|         paused|
|S2A082D3FBC|  U0059|   M0014|     web|       BR|          premium|                33|2026/01/06|     2|      COMPLETED|
|S74DAA8499B|  U0034|   M0048|    NULL|Argentina|             NULL|                93|2026-04-18|     5|      completed|
|SFCF6C0D030|  U0064|   M0016|    NULL| Colômbia|         standard|               179|05-18-2026|     1|           erro|
+-----------+-------+--------+--------+---------+-----------------+------------------+----------+------+---------------+

root
 |-- session_id: string (n

In [0]:
df_logs = transform_logs(df_logs)

print("Logs apos transformacao:")
df_logs.filter(df_logs.session_id.isin('SFCF6C0D030', 'S228044CE8C', 'S2A082D3FBC', 'S74DAA8499B')).sort(df_logs.session_id).show()
df_logs.printSchema()

Logs apos transformacao:
+-----------+-------+--------+--------------+---------+-----------------+------------------+----------+------+---------------+----------------+--------------+
| session_id|user_id|movie_id|        device|  country|subscription_type|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|
+-----------+-------+--------+--------------+---------+-----------------+------------------+----------+------+---------------+----------------+--------------+
|S228044CE8C|  U0036|   M0024|      smart tv|  espanha|             free|                26|2026-02-25|     2|         paused|            0.43|         Curto|
|S2A082D3FBC|  U0059|   M0014|           web|   brasil|          premium|                33|2026-01-06|     2|      completed|            0.55|         Curto|
|S74DAA8499B|  U0034|   M0048|sem informacao|argentina|          unknown|                93|2026-04-18|     5|      completed|            1.55|         Curto|
|SFCF6C0D030|  U0064|

### 1.2 Tratamentos na base de Filmes

In [0]:
import pyspark.sql.functions as f
from itertools import chain
from datetime import datetime

def transform_filmes(df):
    map_genre      = {"action": "ação", "animation": "animação", "comedy": "comédia", "documentary": "documentário"}
    map_expr_genre = f.create_map([f.lit(x) for x in chain(*map_genre.items())])
    map_lang       = {"pt-BR": "pt", "EN-US": "en", "comedy": "comédia", "documentary": "documentário"}
    map_expr_lang  = f.create_map([f.lit(x) for x in chain(*map_lang.items())])
    current_year   = int(datetime.now().year)

    return (
        df
        .dropDuplicates(subset=['movie_id'])
        .withColumn("genre", f.lower(f.trim(f.col("genre"))))
        .withColumn("genre", f.when(f.col("genre").isin(list(map_genre.keys())), map_expr_genre[f.col("genre")]).otherwise(f.col("genre")))
        .withColumn('release_year', f.expr('try_cast(release_year as int)'))
        .withColumn('release_year', f.when(f.col("release_year") <= current_year, f.col("release_year")).otherwise(None))
        .withColumn('duration_minutes', f.expr('try_cast(duration_minutes as int)'))
        .withColumn("content_type", f.lower(f.trim(f.col("content_type"))))
        .withColumn("original_language", f.when(f.col("original_language").isin(list(map_lang.keys())), map_expr_lang[f.col("original_language")])
                    .otherwise(f.col("original_language")))
        .withColumn("title", f.lower(f.trim(f.col("title"))))
        .withColumn("title", f.split(df["title"], r"\s\d+$")[0])
        .fillna("sem informacao")
    )

# df_filmes = transform_filmes(spark.read.csv(f"{files_path}/catalogo_filmes.csv", header=True, inferSchema=True))
# df_filmes.limit(5).show()

In [0]:
df_filmes.filter(df_filmes.movie_id.isin('M0001', 'M0002', 'M0003', 'M0004')).sort(df_filmes.movie_id).show()
df_filmes.printSchema()

+--------+----------------+--------+------------+----------------+------------+-----------------+
|movie_id|           title|   genre|release_year|duration_minutes|content_type|original_language|
+--------+----------------+--------+------------+----------------+------------+-----------------+
|   M0001|Cidade Invisível|Animação|        2000|             163|      series|             NULL|
|   M0002|     Horizonte 2|  Action|        1997|              76|      series|               en|
|   M0003| O Último Trem 3|Animação|        2002|              80|      series|            EN-US|
|   M0004|  Planeta Azul 4|   Drama|        2002|              66|       Movie|            EN-US|
+--------+----------------+--------+------------+----------------+------------+-----------------+

root
 |-- movie_id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- release_year: string (nullable = true)
 |-- duration_minutes: string (nullable = true)
 |--

In [0]:
df_filmes = transform_filmes(df_filmes)

print("Filmes apos transformacao:")
df_filmes.filter(df_filmes.movie_id.isin('M0001', 'M0002', 'M0003', 'M0004')).sort(df_filmes.movie_id).show()
df_filmes.printSchema()

Filmes apos transformacao:
+--------+----------------+--------+------------+----------------+------------+-----------------+
|movie_id|           title|   genre|release_year|duration_minutes|content_type|original_language|
+--------+----------------+--------+------------+----------------+------------+-----------------+
|   M0001|cidade invisível|animação|        2000|             163|      series|   sem informacao|
|   M0002|       horizonte|    ação|        1997|              76|      series|               en|
|   M0003|   o último trem|animação|        2002|              80|      series|               en|
|   M0004|    planeta azul|   drama|        2002|              66|       movie|               en|
+--------+----------------+--------+------------+----------------+------------+-----------------+

root
 |-- movie_id: string (nullable = false)
 |-- title: string (nullable = false)
 |-- genre: string (nullable = false)
 |-- release_year: integer (nullable = true)
 |-- duration_minutes

### 1.3 Tratamento na base de Usuários

In [0]:
import pyspark.sql.functions as f
from itertools import chain

def to_date_multi_format(
    coluna, 
    formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS", "yyyy/dd/MM")
):
    return f.coalesce(*[f.try_to_date(coluna, fmt) for fmt in formatos_suportados])

def transform_usuarios(df):
    mapeamento_country = {"brazil": "brasil", "br": "brasil", "colômbia": "colombia", "spain": "espanha", "méxico": "mexico", "united states": "usa"}
    mapa_expr_country  = f.create_map([f.lit(x) for x in chain(*mapeamento_country.items())])

    return (
        df
        .dropDuplicates(subset=['user_id'])
        .withColumn('age', f.expr('try_cast(age as int)'))
        .withColumn('age', f.when(f.col("age") > 0, f.col("age")).otherwise(None))
        .withColumn("country", f.lower(f.trim(f.col("country"))))
        .withColumn("country", f.when(f.col("country").isin(list(mapeamento_country.keys())), mapa_expr_country[f.col("country")]).otherwise(f.col("country")))
        .withColumn("signup_date", to_date_multi_format(f.split(df["signup_date"], ' ')[0]))
        .withColumn("subscription_type", f.lower(f.trim("subscription_type")))
    )

# df_usuarios = transform_usuarios(spark.read.csv(f"{files_path}/usuarios.csv", header=True, inferSchema=True))
df_usuarios = transform_usuarios(df_usuarios)
df_usuarios.limit(5).show()

+-------+---------------+---+--------+-----------+-----------------+
|user_id|      user_name|age| country|signup_date|subscription_type|
+-------+---------------+---+--------+-----------+-----------------+
|  U0019|     João Souza| 56|     usa| 2024-05-27|         standard|
|  U0024|  Lucas Ruescas| 57|  mexico| 2024-12-31|             NULL|
|  U0034|Nicolas Ruescas| 34|colombia| 2024-04-02|         standard|
|  U0042|   Thiago Costa| 44|  mexico| 2023-03-12|             free|
|  U0046| Julia Oliveira| 16|colombia| 2024-11-10|            trial|
+-------+---------------+---+--------+-----------+-----------------+



In [0]:
df_usuarios.filter(df_usuarios.user_id.isin('U0019', 'U0024', 'U0042', 'U0046')).sort(df_usuarios.user_id).show()
df_filmes.printSchema()

+-------+--------------+---+--------+-----------+-----------------+
|user_id|     user_name|age| country|signup_date|subscription_type|
+-------+--------------+---+--------+-----------+-----------------+
|  U0019|    João Souza| 56|     usa| 2024-05-27|         standard|
|  U0024| Lucas Ruescas| 57|  mexico| 2024-12-31|             NULL|
|  U0042|  Thiago Costa| 44|  mexico| 2023-03-12|             free|
|  U0046|Julia Oliveira| 16|colombia| 2024-11-10|            trial|
+-------+--------------+---+--------+-----------+-----------------+

root
 |-- movie_id: string (nullable = false)
 |-- title: string (nullable = false)
 |-- genre: string (nullable = false)
 |-- release_year: integer (nullable = true)
 |-- duration_minutes: integer (nullable = true)
 |-- content_type: string (nullable = false)
 |-- original_language: string (nullable = false)



In [0]:
df_usuarios = transform_usuarios(df_usuarios)

print("Logs apos transformacao:")
df_usuarios.filter(df_usuarios.user_id.isin('U0019', 'U0024', 'U0042', 'U0046')).sort(df_usuarios.user_id).show()
df_filmes.printSchema()

Logs apos transformacao:
+-------+--------------+---+--------+-----------+-----------------+
|user_id|     user_name|age| country|signup_date|subscription_type|
+-------+--------------+---+--------+-----------+-----------------+
|  U0019|    João Souza| 56|     usa| 2024-05-27|         standard|
|  U0024| Lucas Ruescas| 57|  mexico| 2024-12-31|             NULL|
|  U0042|  Thiago Costa| 44|  mexico| 2023-03-12|             free|
|  U0046|Julia Oliveira| 16|colombia| 2024-11-10|            trial|
+-------+--------------+---+--------+-----------+-----------------+

root
 |-- movie_id: string (nullable = false)
 |-- title: string (nullable = false)
 |-- genre: string (nullable = false)
 |-- release_year: integer (nullable = true)
 |-- duration_minutes: integer (nullable = true)
 |-- content_type: string (nullable = false)
 |-- original_language: string (nullable = false)



### 1.4 Relacionando as bases

In [0]:
def relaciona_bases(bases: list|tuple):
    if bases == []:
        return None
    elif len(bases) == 1:
        return bases[0]
    
    df_rel = bases[0] # spark.createDataFrame(data = [], schema = columns)
    
    for i in range(1,len(bases)):
        campo_ligacao  = [campo for campo in list(set(bases[0].columns).intersection(set(bases[i].columns))) if '_id' in campo]
        campos_remover = [campo for campo in list(set(bases[0].columns).intersection(set(bases[i].columns))) if '_id' not in campo]
        df_rel         = df_rel.drop(*campos_remover).join(bases[i], on=campo_ligacao, how='inner')
    
    print(f"Base final: ({df_rel.count()}, {len(df_rel.columns)})")

    return df_rel

df_final = relaciona_bases([df_logs, df_filmes, df_usuarios])
df_final.limit(5).show()

Base final: (997, 21)
+-------+--------+-----------+--------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+-----------------+--------------+---+---------+-----------+-----------------+
|user_id|movie_id| session_id|        device|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|           title|         genre|release_year|duration_minutes|  content_type|original_language|     user_name|age|  country|signup_date|subscription_type|
+-------+--------+-----------+--------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+-----------------+--------------+---+---------+-----------+-----------------+
|  U0021|   M0010|SD9E82F7AD2|      smart tv|               166|2026-04-23|     1|      buffering|            2.77|        

In [0]:
df_usuarios.drop(*['subscription_type', 'country']).limit(5).show()

+-------+---------------+---+-----------+
|user_id|      user_name|age|signup_date|
+-------+---------------+---+-----------+
|  U0019|     João Souza| 56| 2024-05-27|
|  U0024|  Lucas Ruescas| 57| 2024-12-31|
|  U0034|Nicolas Ruescas| 34| 2024-04-02|
|  U0042|   Thiago Costa| 44| 2023-03-12|
|  U0046| Julia Oliveira| 16| 2024-11-10|
+-------+---------------+---+-----------+



## 2 Consolidando as funções de transformação

In [0]:
import pyspark.sql.functions as f
from itertools import chain
from datetime import datetime

def get_files_content(path):
    df_l = spark.read.csv(f"{path}/logs_streaming.csv", header=True, inferSchema=True)
    df_f = spark.read.csv(f"{path}/catalogo_filmes.csv", header=True, inferSchema=True)
    df_u = spark.read.csv(f"{path}/usuarios.csv", header=True, inferSchema=True)

    # print(f"Logs: ({df_l.count()}, {len(df_l.columns)})\nFilmes: ({df_f.count()}, {len(df_f.columns)})\nUsuários: ({df_u.count()}, {len(df_u.columns)})")

    return df_l, df_f, df_u


def to_date_multi_format(
    coluna, 
    formatos_suportados=("MM/dd/yyyy", "yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy", "yyyy-MM-dd HH:MM:SS", "yyyy/dd/MM")
):
    return f.coalesce(*[f.try_to_date(coluna, fmt) for fmt in formatos_suportados])


def transform_logs(df):
    mapeamento_status  = {"complete": "completed", "erro": "error"}
    mapa_expr_status   = f.create_map([f.lit(x) for x in chain(*mapeamento_status.items())])
    mapeamento_country = {"brazil": "brasil", "br": "brasil", "colômbia": "colombia", "spain": "espanha", "méxico": "mexico", "united states": "usa"}
    mapa_expr_country  = f.create_map([f.lit(x) for x in chain(*mapeamento_country.items())])

    return (
        df
        .dropDuplicates(subset=['session_id', 'user_id', 'movie_id'])
        .withColumn('watch_time_minutes', f.expr('try_cast(watch_time_minutes as float)').cast('int'))
        .withColumn('watch_time_minutes', f.when(f.col("watch_time_minutes") > 0, f.col("watch_time_minutes")).otherwise(0))
        .withColumn("watch_time_hours", f.round(f.col("watch_time_minutes")/60, 2))
        .withColumn("watch_category", f.when((f.col("watch_time_minutes") >= 120) & (df.watch_time_minutes.isNotNull()), "Longo")
                    .otherwise(f.when(f.col("watch_time_minutes") < 120, "Curto").otherwise(None)))
        .withColumn("playback_status", f.lower(f.trim(f.col("playback_status"))))
        .withColumn("playback_status", f.when(f.col("playback_status").isin(list(mapeamento_status.keys())), mapa_expr_status[f.col("playback_status")])
                    .otherwise(f.col("playback_status")))
        .withColumn("subscription_type", f.lower("subscription_type"))
        .withColumn("country", f.lower(f.trim(f.col("country"))))
        .withColumn("country", f.when(f.col("country").isin(list(mapeamento_country.keys())), mapa_expr_country[f.col("country")]).otherwise(f.col("country")))
        .withColumn("device", f.lower(f.trim(f.col("device"))))
        .withColumn('device', f.regexp_replace('device', '_', ' '))
        .withColumn('rating', f.expr('try_cast(rating as float)').cast('int'))
        .withColumn('rating', f.when(f.col("rating") > 5, 5).otherwise(f.col("rating")))
        .withColumn("watch_date", to_date_multi_format(f.split(df["watch_date"], ' ')[0]))
        .dropna(subset=['user_id']).dropna(subset=['movie_id']).dropna(subset=['session_id'])
        .fillna({"country": "desconhecido", "subscription_type": "unknown"})
        .fillna("sem informacao")
    )

def transform_filmes(df):
    map_genre      = {"action": "ação", "animation": "animação", "comedy": "comédia", "documentary": "documentário"}
    map_expr_genre = f.create_map([f.lit(x) for x in chain(*map_genre.items())])
    map_lang       = {"pt-BR": "pt", "EN-US": "en", "comedy": "comédia", "documentary": "documentário"}
    map_expr_lang  = f.create_map([f.lit(x) for x in chain(*map_lang.items())])
    current_year   = int(datetime.now().year)

    return (
        df
        .dropDuplicates(subset=['movie_id'])
        .withColumn("genre", f.lower(f.trim(f.col("genre"))))
        .withColumn("genre", f.when(f.col("genre").isin(list(map_genre.keys())), map_expr_genre[f.col("genre")]).otherwise(f.col("genre")))
        .withColumn('release_year', f.expr('try_cast(release_year as int)'))
        .withColumn('release_year', f.when(f.col("release_year") <= current_year, f.col("release_year")).otherwise(None))
        .withColumn('duration_minutes', f.expr('try_cast(duration_minutes as int)'))
        .withColumn("content_type", f.lower(f.trim(f.col("content_type"))))
        .withColumn("original_language", f.when(f.col("original_language").isin(list(map_lang.keys())), map_expr_lang[f.col("original_language")])
                    .otherwise(f.col("original_language")))
        .withColumn("title", f.lower(f.trim(f.col("title"))))
        .withColumn("title", f.split(df["title"], r"\s\d+$")[0])
        .fillna("sem informacao")
    )

def transform_usuarios(df):
    mapeamento_country = {"brazil": "brasil", "br": "brasil", "colômbia": "colombia", "spain": "espanha", "méxico": "mexico", "united states": "usa"}
    mapa_expr_country  = f.create_map([f.lit(x) for x in chain(*mapeamento_country.items())])

    return (
        df
        .dropDuplicates(subset=['user_id'])
        .withColumn('age', f.expr('try_cast(age as int)'))
        .withColumn('age', f.when(f.col("age") > 0, f.col("age")).otherwise(None))
        .withColumn("country", f.lower(f.trim(f.col("country"))))
        .withColumn("country", f.when(f.col("country").isin(list(mapeamento_country.keys())), mapa_expr_country[f.col("country")]).otherwise(f.col("country")))
        .withColumn("signup_date", to_date_multi_format(f.split(df["signup_date"], ' ')[0]))
        .withColumn("subscription_type", f.lower(f.trim("subscription_type")))
    )

def relaciona_bases(bases: list|tuple):
    if bases == []:
        return None
    elif len(bases) == 1:
        return bases[0]
    
    df_rel = bases[0] # spark.createDataFrame(data = [], schema = columns)
    
    for i in range(1,len(bases)):
        campo_ligacao  = [campo for campo in list(set(bases[0].columns).intersection(set(bases[i].columns))) if '_id' in campo]
        campos_remover = [campo for campo in list(set(bases[0].columns).intersection(set(bases[i].columns))) if '_id' not in campo]
        df_rel         = df_rel.drop(*campos_remover).join(bases[i], on=campo_ligacao, how='inner')
    
    # print(f"Base final: ({df_rel.count()}, {len(df_rel.columns)})") # apenas para dev, para nao executar as transformacoes

    return df_rel


df_logs, df_filmes, df_usuarios = get_files_content(files_path)
df_logs                         = transform_logs(df_logs)
df_filmes                       = transform_filmes(df_filmes)
df_usuarios                     = transform_usuarios(df_usuarios)
df_final                        = relaciona_bases([df_logs, df_filmes, df_usuarios])
df_final.limit(5).show()

+-------+--------+-----------+--------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+-----------------+--------------+---+---------+-----------+-----------------+
|user_id|movie_id| session_id|        device|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|           title|         genre|release_year|duration_minutes|  content_type|original_language|     user_name|age|  country|signup_date|subscription_type|
+-------+--------+-----------+--------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+-----------------+--------------+---+---------+-----------+-----------------+
|  U0021|   M0010|SD9E82F7AD2|      smart tv|               166|2026-04-23|     1|      buffering|            2.77|         Longo|       horizont

## 3 Performance

### 3.1 Cache

O Spark possui um mecanismo chamado "_lazy evaluation/lazy processing_", que faz com que as operações de transformação definidas nos DataFrames não sejam executadas até que uma "_action_" seja chamada para esse DataFrame.

Como exemplos de transformações temos: `filter()`, `groupBy()`, `withColumn()`, `dropDuplicates()`, `fillna()` `select()` e `join()`. E de _actions_: `count()`, `display()`, `show()`, `write()`, `collect()`.

Conforme vamos definindo essas operações, o Spark monta um "plano de execução" (DAG, directed acyclic graph), mas não o executa. Quando uma _action_ é chamada, o Spark otimiza o plano de execução e executa todas as operações de transformação necessárias para produzir o resultado dessa _action_. 

Mas digamos que você precise executar diferentes _actions_ em um mesmo DataFrame resultante, nesse caso, a cada _action_ chamada o DataFrame será "reconstruido". Para evitar isso, podemos usar o recurso de "_caching_", em que dizemos ao Spark para manter em memória o resultado daquele DataFrame. Dessa forma, esse ciclo de recomputações é "quebrado" e quando uma outra _action_ for chamada para o DataFrame não será necessário executar todas as transformações novamente.

Para cachear o resultado de um DataFrame na memória podemos usar dois métodos: `cache()` e `persist()`, logo antes de chamar uma _action_. Ou seja, os métodos de cache() apenas marcam o DataFrame para ser cacheado, mas eles também precisam de uma ação para serem materializados em memória.

```python
df_final = relaciona_bases(get_files_content())                 # define todas as transformacoes nas bases e as relaciona montando o DataFrame final
df_final.cache()                                                # marca DataFrame final para ser cacheado em memoria
print(f"Base final: ({df_rel.count()}, {len(df_rel.columns)})") # executa as tranformacoes e armazena resultado em memoria

# executa transformacoes no DataFrame df_final, mas sem reprocessa-lo a cada nova transformacao no resultado final
(df_final.groupBy('genre')
         .agg(f.count('*').alias("total_views"),
              f.round(f.sum("watch_time_hours"), 2).alias("total_watch_time_hours"),
              f.round(f.avg("watch_time_minutes"), 2).alias("average_watch_time_minutes"))
         .sort('total_views', ascending=False)).show()

(df_final.groupBy('country')
         .agg(f.count('*').alias("total_views"),
              f.round(f.sum("watch_time_hours"), 2).alias("total_watch_time_hours"),
              f.round(f.avg("watch_time_minutes"), 2).alias("average_watch_time_minutes"))
         .sort('total_views', ascending=False)).show()

(df_final.groupBy('subscription_type')
         .agg(f.count('*').alias("total_views"),
              f.round(f.sum("watch_time_hours"), 2).alias("total_watch_time_hours"),
              f.round(f.avg("watch_time_minutes"), 2).alias("average_watch_time_minutes"))
         .sort('total_views', ascending=False)).show()

df_final.unpersist() # remove DataFrame da memoria
```

In [0]:
(df_final.groupBy('genre')
         .agg(f.count('*').alias("total_views"),
              f.round(f.sum("watch_time_hours"), 2).alias("total_watch_time_hours"),
              f.round(f.avg("watch_time_minutes"), 2).alias("average_watch_time_minutes"))
         .sort('total_views', ascending=False)).show()

+--------------+-----------+----------------------+--------------------------+
|         genre|total_views|total_watch_time_hours|average_watch_time_minutes|
+--------------+-----------+----------------------+--------------------------+
|          ação|        179|                904.89|                    303.31|
|sem informacao|        162|                 738.6|                    273.56|
|      animação|        131|                339.49|                    155.53|
|  documentário|        112|                330.67|                    177.13|
|       comédia|         86|                122.66|                     85.56|
|      suspense|         84|                628.02|                     448.6|
|        horror|         63|                  94.5|                     89.97|
|       romance|         57|                 87.56|                     92.18|
|        terror|         52|                 405.2|                    467.56|
|        sci-fi|         38|                217.81| 

In [0]:
(df_final.groupBy('country')
         .agg(f.count('*').alias("total_views"),
              f.round(f.sum("watch_time_hours"), 2).alias("total_watch_time_hours"),
              f.round(f.avg("watch_time_minutes"), 2).alias("average_watch_time_minutes"))
         .sort('total_views', ascending=False)).show()

+---------+-----------+----------------------+--------------------------+
|  country|total_views|total_watch_time_hours|average_watch_time_minutes|
+---------+-----------+----------------------+--------------------------+
|      usa|        164|                233.67|                     85.48|
| colombia|        164|               1224.73|                    448.09|
|   mexico|        155|                377.66|                    146.19|
|   brasil|        148|                552.81|                    224.11|
|argentina|        147|                716.91|                    292.63|
|  espanha|        126|                682.75|                    325.12|
| portugal|         65|                 95.33|                      88.0|
|    chile|         28|                  41.0|                     87.86|
+---------+-----------+----------------------+--------------------------+



In [0]:
(df_final.groupBy('subscription_type')
         .agg(f.count('*').alias("total_views"),
              f.round(f.sum("watch_time_hours"), 2).alias("total_watch_time_hours"),
              f.round(f.avg("watch_time_minutes"), 2).alias("average_watch_time_minutes"))
         .sort('total_views', ascending=False)).show()

+-----------------+-----------+----------------------+--------------------------+
|subscription_type|total_views|total_watch_time_hours|average_watch_time_minutes|
+-----------------+-----------+----------------------+--------------------------+
|            basic|        326|               1631.74|                    300.31|
|          premium|        267|               1367.99|                    307.43|
|            trial|        116|                171.37|                     88.63|
|             NULL|        110|                496.11|                    270.62|
|         standard|         91|                130.59|                     86.11|
|             free|         87|                127.06|                     87.62|
+-----------------+-----------+----------------------+--------------------------+



#### Usando `cache()` dentro de funções

Um ponto interessante sobre o cache que vale mencionar é sobre a utilização dentro de funções. Vamos usar como exemplo a função `relaciona_bases()`, que criamos para relacionar as três bases que estamos trabalhando. 

Se realizarmos o cache do DataFrame que será retornado (chamando uma _action_ logo depois), então o DataFrame retornado pela função estará cacheado, mas seu comportamento não está restrito ao escopo da função, como é o caso de variáveis. O DataFrame é materializado em memória e vai existir lá até que seja removido usando o método `unpersist()` ou o _cluster_ seja desligado. 

O cache funciona para o DataFrame e não para o retorno da função. O Spark não sabe se a função já foi executada ou não, ele vai apenas colocar mais um DataFrame na memória (se houver espaço). Portanto, é necessário ter cuidado ao utilizar cache dentro de funções.

```python
def relaciona_bases(bases: list|tuple):
    if bases == []:
        return None
    df_rel = bases[0]
    
    for i in range(1,len(bases)):
        campo_ligacao  = [campo for campo in list(set(bases[0].columns).intersection(set(bases[i].columns))) if '_id' in campo]
        campos_remover = [campo for campo in list(set(bases[0].columns).intersection(set(bases[i].columns))) if '_id' not in campo]
        df_rel         = df_rel.drop(*campos_remover).join(bases[i], on=campo_ligacao, how='inner')
    
    df_rel.cache()
    print(f"Base final: ({df_rel.count()}, {len(df_rel.columns)})") # apenas para dev, para nao executar as transformacoes
    return df_rel

# chama funcao pela primeira vez, executa as transformacoes necessarias para chegar em `df_final_1` e o armazena na memporia
df_final_1 = relaciona_bases([df_logs, df_filmes, df_usuarios])

# chama funcao pela segunda vez, executa novamente todas transformacoes, retorna `df_final_2` e o armazena na memporia
# df_final_2 é um novo DataFrame, identico a `df_final_1`, e tambem ocupa espaco na memporia
df_final_2 = relaciona_bases([df_logs, df_filmes, df_usuarios]) 
```

### 3.2 Particionamento

O particionamento é a forma como o Spark organiza os dados para possibilitar o uso de processamento paralelo de forma eficiente. Ao dividir o `df_final` em quatro partições com `repartition(4)` estamos dizendo ao Spark para organizar esses dados de forma que ele possa distribuí-los entre quatro "executores" para processamento paralelo. Nesse caso em específico, são criados quatro arquivos, cada um deles contendo uma parte dos dados.

Essa é uma forma de otimizar a leitura de dados: ao carregar uma pasta que contém arquivos Parquet, o Spark automaticamente reconhece que são partições de um mesmo conjunto de dados. Ele então distribuirá a leitura e o processamento desses quatro arquivos entre os recursos disponíveis, aproveitando o paralelismo. Dessa forma os dados são lidos muito mais rapidamente do que se estivéssemos lendo um único arquivo CSV, por exemplo.

In [0]:
# particiona base final em 4 partes (sem criterio) e armazena em formato parquet
df_final_reparticionado_4p  = df_final.repartition(4)
path_arquivos_particionados = f"{files_path}/base_final_reparticionada_sem_criterio"
df_final_reparticionado_4p.write.mode("overwrite").parquet(path_arquivos_particionados)

In [0]:
%%time
df_parquet = spark.read.parquet(path_arquivos_particionados)
df_parquet.limit(5).show()

+-------+--------+-----------+--------------+------------------+----------+------+---------------+----------------+--------------+----------------+------+------------+----------------+------------+-----------------+-------------+---+-------+-----------+-----------------+
|user_id|movie_id| session_id|        device|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|           title| genre|release_year|duration_minutes|content_type|original_language|    user_name|age|country|signup_date|subscription_type|
+-------+--------+-----------+--------------+------------------+----------+------+---------------+----------------+--------------+----------------+------+------------+----------------+------------+-----------------+-------------+---+-------+-----------+-----------------+
|  U0001|   M0002|SA045C23AEC|      smart tv|                 2|2026-04-06|     2| sem informacao|            0.03|         Curto|       horizonte|  ação|        1997|              76|

In [0]:
# salva o conteudo de df_final em arquivo CSV
path_csv = f"{files_path}/base_final_csv"
df_final.write.mode("overwrite").option("header", True).csv(path_csv)

In [0]:
%%time
df_csv = spark.read.csv(path_csv, header=True, inferSchema=True)
df_csv.limit(5).show()

+-------+--------+-----------+--------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+-----------------+--------------+---+---------+-----------+-----------------+
|user_id|movie_id| session_id|        device|watch_time_minutes|watch_date|rating|playback_status|watch_time_hours|watch_category|           title|         genre|release_year|duration_minutes|  content_type|original_language|     user_name|age|  country|signup_date|subscription_type|
+-------+--------+-----------+--------------+------------------+----------+------+---------------+----------------+--------------+----------------+--------------+------------+----------------+--------------+-----------------+--------------+---+---------+-----------+-----------------+
|  U0021|   M0010|SD9E82F7AD2|      smart tv|               166|2026-04-23|     1|      buffering|            2.77|         Longo|       horizont

In [0]:
# particiona base final de acordo com o genero dos filmes
df_final_reparticionado_genre = df_final.repartition("genre")
df_final_reparticionado_genre.write.mode("overwrite").parquet(f"{files_path}/base_final_reparticionada_genre")